In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuration
mne.set_log_level('WARNING')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Dossiers
Path("figures").mkdir(exist_ok=True)
Path("reports").mkdir(exist_ok=True)

print("✅ Bibliothèques chargées")
print(f"MNE version: {mne.__version__}")

✅ Bibliothèques chargées
MNE version: 1.11.0


In [3]:
# ============================================================================
# CHARGEMENT DU SUJET SAUVEGARDÉ (à partir du Notebook 01)
# ============================================================================

print("\n" + "="*60)
print("CHARGEMENT DES DONNÉES")
print("="*60)

# Essayer de charger depuis le dossier reports
try:
    raw = mne.io.read_raw_fif('reports/raw_subject_1_preprocessed.fif', 
                              preload=True, 
                              verbose=False)
    print("✅ Sujet chargé depuis reports/raw_subject_1_preprocessed.fif")
except FileNotFoundError:
    print("⚠️ Fichier non trouvé, chargement depuis PhysioNet...")
    
    # Si le fichier n'existe pas, on recharge depuis PhysioNet
    subjects = [1]
    runs = [6, 10, 14]
    
    raw_fnames = mne.datasets.eegbci.load_data(1, runs, verbose=False)
    raws = []
    for fname in raw_fnames:
        raw_temp = mne.io.read_raw_edf(fname, preload=True, verbose=False)
        raws.append(raw_temp)
    
    raw = mne.concatenate_raws(raws, verbose=False)
    mne.datasets.eegbci.standardize(raw)
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage, verbose=False)
    
    print("✅ Sujet chargé depuis PhysioNet")

print(f"   - Durée : {raw.times[-1]:.1f} secondes")
print(f"   - Canaux : {len(raw.ch_names)}")
print(f"   - Fréquence : {raw.info['sfreq']} Hz")


CHARGEMENT DES DONNÉES
⚠️ Fichier non trouvé, chargement depuis PhysioNet...


StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.

In [ ]:
#Filtrage passe-bande 1-40 Hz

print("\n" + "="*60)
print("ÉTAPE 1 : FILTRAGE PASSE-BANDE (1-40 Hz)")
print("="*60)

raw_filtered = raw.copy()

raw_filtered.filter(
    l_freq=1.0,
    h_freq=40.0,
    method='fir',
    fir_design='firwin2',
    verbose=False
)

print("✅ Filtre passe-bande 1-40 Hz appliqué")
print("   - Fréquences < 1 Hz (dérive) : éliminées")
print("   - Fréquences > 40 Hz (EMG) : éliminées")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

psd_before = raw.copy().pick_types(eeg=True).compute_psd(fmin=0, fmax=50, verbose=False)
psd_before.plot(axes=axes[0], average=True, show=False)
axes[0].set_title('PSD - AVANT filtrage')

psd_after = raw_filtered.copy().pick_types(eeg=True).compute_psd(fmin=0, fmax=50, verbose=False)
psd_after.plot(axes=axes[1], average=True, show=False)
axes[1].set_title('PSD - APRÈS filtrage (1-40 Hz)')

plt.tight_layout()
plt.savefig('figures/filtering_comparison.png', dpi=150)
plt.show()

print("✅ Figure sauvegardée : figures/filtering_comparison.png")

In [ ]:
# ============================================================================
# ÉTAPE 2 : NOTCH FILTER 50 Hz (SUPPRESSION BRUIT SECTEUR)
# ============================================================================

print("\n" + "="*60)
print("ÉTAPE 2 : NOTCH FILTER 50 Hz")
print("="*60)

# Pourquoi 50 Hz ? C'est la fréquence du réseau électrique (Europe, Afrique)
# Ce bruit est capté par les électrodes comme une antenne

raw_notched = raw_filtered.copy()

# Filtre coupe-bande à 50 Hz
raw_notched.notch_filter(
    freqs=50.0,          # Fréquence à couper
    picks='eeg',         # Appliquer seulement sur les canaux EEG
    method='fir',
    verbose=False
)

print("✅ Notch filter 50 Hz appliqué")
print("   - Supprime le bourdonnement du réseau électrique")
print("   - N'altère pas les fréquences physiologiques voisines (49-51 Hz épargnés)")

# Visualisation de l'effet du notch
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Avant notch (juste après passe-bande)
psd_before = raw_filtered.copy().pick_types(eeg=True).compute_psd(fmin=45, fmax=55, verbose=False)
psd_before.plot(axes=axes[0], average=True, show=False)
axes[0].set_title('PSD - AVANT Notch (45-55 Hz)')
axes[0].axvline(50, color='red', linestyle='--', alpha=0.5, label='50 Hz')
axes[0].legend()

# Après notch
psd_after = raw_notched.copy().pick_types(eeg=True).compute_psd(fmin=45, fmax=55, verbose=False)
psd_after.plot(axes=axes[1], average=True, show=False)
axes[1].set_title('PSD - APRÈS Notch (45-55 Hz)')
axes[1].axvline(50, color='red', linestyle='--', alpha=0.5, label='50 Hz (supprimé)')
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/notch_filter_effect.png', dpi=150)
plt.show()

print("✅ Figure sauvegardée : figures/notch_filter_effect.png")
print("\n➡️ Tu devrais voir un 'trou' (atténuation) autour de 50 Hz !")

In [ ]:
# ============================================================================
# ÉTAPE 3 : AVERAGE REFERENCE
# ============================================================================

print("\n" + "="*60)
print("ÉTAPE 3 : AVERAGE REFERENCE")
print("="*60)

# Pourquoi ? Le signal EEG est toujours mesuré par rapport à une référence
# (ex: électrode sur l'oreille). La référence moyenne rend le signal
# indépendant du choix de référence initial.

raw_referenced = raw_notched.copy()

# Appliquer la référence moyenne
raw_referenced.set_eeg_reference('average', projection=False, verbose=False)

print("✅ Average reference appliqué")
print("   - La moyenne de tous les canaux devient la nouvelle référence")
print("   - Rend le système standardisé entre sujets")

# Visualisation : comparer un canal avant/après
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Sélectionner un canal moteur (C3)
times = raw_referenced.times[:int(3 * raw_referenced.info['sfreq'])]  # 3 secondes

# Avant référence
data_before = raw_notched.get_data(picks=['C3'])[0][:len(times)]
axes[0].plot(times, data_before, 'b-', linewidth=0.8)
axes[0].set_title('Canal C3 - AVANT average reference')
axes[0].set_ylabel('Amplitude (µV)')
axes[0].set_xlabel('Temps (s)')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)

# Après référence
data_after = raw_referenced.get_data(picks=['C3'])[0][:len(times)]
axes[1].plot(times, data_after, 'g-', linewidth=0.8)
axes[1].set_title('Canal C3 - APRÈS average reference')
axes[1].set_ylabel('Amplitude (µV)')
axes[1].set_xlabel('Temps (s)')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.savefig('figures/average_reference_effect.png', dpi=150)
plt.show()

print("✅ Figure sauvegardée : figures/average_reference_effect.png")
print("\n📊 Ce que tu observes :")
print("   - Le signal est désormais centré autour de zéro")
print("   - L'amplitude absolue n'a plus de sens, seules les différences comptent")

In [ ]:
# ============================================================================
# ÉTAPE 4 : ICA - CALCUL DES COMPOSANTES
# ============================================================================

print("\n" + "="*60)
print("ÉTAPE 4 : INDEPENDENT COMPONENT ANALYSIS (ICA)")
print("="*60)

"""
CONCEPT SIMPLE :
Le signal EEG à la surface = mélange de plusieurs sources :
- Source 1 : activité cérébrale (ce qu'on veut)
- Source 2 : clignements des yeux (EOG) (ce qu'on veut enlever)
- Source 3 : bruit musculaire (EMG) (ce qu'on veut enlever)
- Source 4 : bruit du secteur (ce qu'on veut enlever)

ICA est comme un algorithme qui remixe l'audio pour séparer chaque source.
"""

from mne.preprocessing import ICA

print("\n🔄 Calcul de l'ICA (cela peut prendre 30-60 secondes)...")

# On travaille sur le signal filtré (pas encore référencé pour ICA)
raw_ica = raw_filtered.copy()

# Configuration ICA
ica = ICA(
    n_components=20,      # Extraire 20 sources indépendantes
    random_state=42,      # Pour résultats reproductibles
    method='fastica',     # Algorithme rapide
    verbose=False
)

# Calculer l'ICA
ica.fit(raw_ica, picks='eeg', verbose=False)

print("✅ ICA calculé avec succès !")
print(f"   - {ica.n_components_} composantes extraites")

# Sauvegarder l'ICA pour reuse plus tard
ica.save('reports/ica_components.fif', overwrite=True)
print("✅ ICA sauvegardé : reports/ica_components.fif")

# Visualisation des topographies des 20 composantes
print("\n📊 Visualisation des composantes...")

ica.plot_components(picks=range(20), show=False)
plt.savefig('figures/ica_all_components.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Figure sauvegardée : figures/ica_all_components.png")

In [ ]:
# ============================================================================
# ÉTAPE 4 : ICA - IDENTIFICATION DES ARTEFACTS OCULAIRES (EOG)
# ============================================================================

print("\n" + "="*60)
print("IDENTIFICATION DES COMPOSANTES EOG")
print("="*60)

# Trouver les canaux EOG dans l'enregistrement
eog_channels = mne.pick_types(raw_ica.info, meg=False, eog=True)

if len(eog_channels) > 0:
    print(f"\n👁️ Canaux EOG trouvés :")
    for idx in eog_channels:
        print(f"   - {raw_ica.ch_names[idx]}")
    
    # Calculer automatiquement la corrélation avec les canaux EOG
    eog_indices, eog_scores = ica.find_bads_eog(
        raw_ica,
        ch_name=eog_channels[0],  # Premier canal EOG
        threshold=3.0,             # Seuil de corrélation
        verbose=False
    )
    
    print(f"\n📊 Composantes corrélées avec les EOG : {eog_indices}")
    
    if len(eog_indices) > 0:
        # Visualiser les composantes suspectes
        ica.plot_properties(raw_ica, picks=eog_indices, verbose=False, show=False)
        plt.savefig('figures/ica_eog_components.png', dpi=150, bbox_inches='tight')
        plt.close()
        print("✅ Figure sauvegardée : figures/ica_eog_components.png")
else:
    print("\n⚠️ Aucun canal EOG trouvé dans les métadonnées")
    print("   Il faudra identifier manuellement les composantes oculaires")

print("\n🔍 Guide d'identification visuelle :")
print("   - Composante EOG typique :")
print("     * Topographie : pic sur les électrodes frontales (Fp1, Fp2, Fz)")
print("     * Time-series : pics brusques et amples (200-300 µV)")
print("     * Spectre : dominante dans delta (<4 Hz)")

# Visualisation interactive (décommente si besoin)
# ica.plot_components(picks=range(10))
# ica.plot_sources(raw_ica, start=0, duration=10)

# Exemple d'exclusion manuelle (à adapter après visualisation)
# ica.exclude = [0]  # Exclure la composante 0 si c'est un EOG
# raw_clean = ica.apply(raw_referenced.copy())

In [ ]:
# ============================================================================
# ÉTAPE 4 (suite) : APPLICATION DE L'ICA — SUPPRESSION DES ARTEFACTS
# ============================================================================

print("\n" + "="*60)
print("APPLICATION DE L'ICA — SUPPRESSION DES ARTEFACTS")
print("="*60)

# Si aucune composante EOG trouvée automatiquement, exclure manuellement la 0
if len(eog_channels) == 0 or len(eog_indices) == 0:
    print("⚠️ Aucune composante EOG détectée automatiquement")
    print("   → Exclusion manuelle de la composante 0 (souvent oculaire)")
    ica.exclude = [0]
else:
    ica.exclude = eog_indices
    print(f"✅ Composantes à exclure : {ica.exclude}")

# Appliquer l'ICA sur le signal référencé (raw_referenced)
raw_clean = ica.apply(raw_referenced.copy(), verbose=False)
print(f"✅ ICA appliqué — {len(ica.exclude)} composante(s) supprimée(s)")

# ── Comparaison avant/après ICA sur canal C3 ──────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
n_samples = int(5 * raw_referenced.info['sfreq'])  # 5 secondes
times_plot = raw_referenced.times[:n_samples]

axes[0].plot(times_plot, raw_referenced.get_data(picks=['C3'])[0][:n_samples],
             'r-', linewidth=0.8)
axes[0].set_title('C3 — AVANT ICA (artefacts oculaires visibles)')
axes[0].set_ylabel('Amplitude (µV)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(times_plot, raw_clean.get_data(picks=['C3'])[0][:n_samples],
             'g-', linewidth=0.8)
axes[1].set_title('C3 — APRÈS ICA (artefacts supprimés)')
axes[1].set_ylabel('Amplitude (µV)')
axes[1].set_xlabel('Temps (s)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/ica_before_after.png', dpi=150)
plt.show()
print("✅ Figure sauvegardée : figures/ica_before_after.png")


In [ ]:
# ============================================================================
# ÉTAPE 5 : EPOCHING - DÉCOUPAGE EN SEGMENTS TEMPORELS
# ============================================================================

print("\n" + "="*60)
print("ÉTAPE 5 : EPOCHING")
print("="*60)

"""
CONCEPT SIMPLE :

Epoching = découper le signal continu en petits segments
Chaque segment = un essai (une tentative d'imagerie motrice)

TIMING TYPIQUE :
- tmin = -1s : 1 seconde AVANT l'instruction (baseline/repos)
- t = 0s : début de l'instruction "imaginez bouger la main"
- tmax = +4s : 4 secondes APRÈS (pendant l'imagerie)

Pourquoi baseline (-1s à 0) ? Pour normaliser le signal :
epoch_normalisé = (epoch - baseline_moyen) / baseline_std
"""

# Extraire les événements depuis le signal
events, event_id = mne.events_from_annotations(raw_clean, verbose=False)

print(f"\n🎯 Événements trouvés : {len(events)}")
print(f"   - Codes disponibles : {event_id}")

# Paramètres d'epoching
tmin = -1.0   # Début : 1 seconde avant l'événement (capture le repos)
tmax = 4.0    # Fin : 4 secondes après (durée de l'imagerie)
baseline = (-1.0, 0)  # Période de référence (juste avant l'instruction)

print(f"\n⏱️  Paramètres :")
print(f"   - Fenêtre temporelle : [{tmin}, {tmax}] secondes")
print(f"   - Baseline : {baseline} (pour normalisation)")
print(f"   - Durée d'une epoch : {tmax - tmin} secondes")
print(f"   - Points par epoch : {int((tmax - tmin) * raw_referenced.info['sfreq'])}")

# Création des epochs
# IMPORTANT : on epoch sur raw_clean (après ICA), pas raw_referenced
epochs = mne.Epochs(
    raw_clean,
    events,
    event_id,
    tmin=tmin,
    tmax=tmax,
    baseline=baseline,
    picks='eeg',          # Ne garder que les canaux EEG
    preload=True,         # Charger en mémoire pour accès rapide
    reject=dict(eeg=500e-6),  # Seuil 500 µV
    verbose=False
)

print(f"\n✅ Epochs créées avec succès !")
print(f"   - Nombre total : {len(epochs)}")
print(f"   - Shape : {epochs.get_data().shape}")
print(f"     (epochs × canaux × temps)")

# Distribution par classe
print(f"\n📊 Distribution par classe :")
for event_name, event_code in event_id.items():
    n_epochs = np.sum(epochs.events[:, -1] == event_code)
    print(f"   - {event_name} (code {event_code}): {n_epochs} epochs")



# Séparer les 3 classes
epochs_left  = epochs['T1']   # main gauche
epochs_right = epochs['T2']   # main droite
epochs_both  = epochs['T0']   # repos / both hands
print(f"\n✅ Classes séparées : {len(epochs_left)} gauche | "
      f"{len(epochs_right)} droite | {len(epochs_both)} repos")
# Visualisation d'une epoch seule
fig = epochs[0].plot(
    n_channels=10,
    scalings='auto',
    title=f'Exemple d\'epoch - Classe: {list(event_id.keys())[0]}',
    show=False
)
plt.savefig('figures/single_epoch_example.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n✅ Figure sauvegardée : figures/single_epoch_example.png")

In [ ]:
# Diagnostic : voir quelles variables existent
print("🔍 Diagnostic des variables :")
print(f"   raw : {'✅ défini' if 'raw' in dir() else '❌ non défini'}")
print(f"   raw_filtered : {'✅ défini' if 'raw_filtered' in dir() else '❌ non défini'}")
print(f"   raw_notched : {'✅ défini' if 'raw_notched' in dir() else '❌ non défini'}")
print(f"   raw_referenced : {'✅ défini' if 'raw_referenced' in dir() else '❌ non défini'}")
print(f"   epochs : {'✅ défini' if 'epochs' in dir() else '❌ non défini'}")
print(f"   epochs_clean : {'✅ défini' if 'epochs_clean' in dir() else '❌ non défini'}")
print(f"   evoked_left : {'✅ défini' if 'evoked_left' in dir() else '❌ non défini'}")

In [ ]:
# ============================================================================
# DIAGNOSTIC : Pourquoi toutes les epochs sont rejetées ?
# ============================================================================

print("\n" + "="*60)
print("DIAGNOSTIC DU REJET")
print("="*60)

# Afficher les raisons de rejet
print("\n📋 Raisons de rejet (5 premières epochs) :")
for i, log in enumerate(epochs.drop_log[:10]):
    if len(log) > 0:
        print(f"   Epoch {i}: {log}")

# Visualiser les drop logs
fig = epochs.plot_drop_log(show=False)
fig.savefig('figures/drop_log_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n✅ Figure sauvegardée : figures/drop_log_analysis.png")

# Vérifier l'amplitude max des époques
print("\n📊 Amplitude maximale par epoch :")
max_amps = []
for i in range(min(10, len(epochs))):
    data = epochs.get_data()[i]
    max_amp = np.max(np.abs(data)) * 1e6  # Conversion en µV
    max_amps.append(max_amp)
    print(f"   Epoch {i}: {max_amp:.1f} µV")

print(f"\n   Amplitude moyenne max : {np.mean(max_amps):.1f} µV")
print(f"   Seuil actuel : 150 µV")

# Suggestion
if np.mean(max_amps) > 150:
    print("\n⚠️ Les amplitudes dépassent 150 µV → Il faut AUGMENTER le seuil !")
    suggested_threshold = int(np.mean(max_amps) * 1.2)
    print(f"   Seuil suggéré : {suggested_threshold} µV")

In [ ]:
# ============================================================================
# ÉTAPE 6 CORRIGÉE : Rejet avec seuil adapté
# ============================================================================

print("\n" + "="*60)
print("ÉTAPE 6 : REJET DES ARTEFACTS (SEUIL ADAPTÉ)")
print("="*60)

# Calculer un seuil adapté (par exemple 500 µV au lieu de 150 µV)
# Pourquoi 500 µV ? C'est une valeur standard pour l'EEG de scalp
reject_criteria = dict(eeg=500e-6)  # 500 microvolts (plus réaliste)

print(f"📏 Seuil : amplitude pic-à-pic > 500 µV (ajusté)")

n_before = len(epochs)
print(f"\n📊 Avant rejet : {n_before} epochs")

# Appliquer le rejet
epochs_clean = epochs.copy().drop_bad(reject=reject_criteria, verbose=False)

n_after = len(epochs_clean)
n_rejected = n_before - n_after
rejection_rate = (n_rejected / n_before) * 100

print(f"\n📊 Après rejet : {n_after} epochs")
print(f"   - Rejetées : {n_rejected} ({rejection_rate:.1f}%)")
print(f"   - Conservées : {n_after / n_before * 100:.1f}%")

if n_after == 0:
    print("\n⚠️ ENCORE REJET TOTAL ! Tentons 1000 µV...")
    reject_criteria = dict(eeg=1000e-6)
    epochs_clean = epochs.copy().drop_bad(reject=reject_criteria, verbose=False)
    n_after = len(epochs_clean)
    print(f"   Avec seuil 1000 µV : {n_after} epochs conservées")

if n_after > 0:
    # Distribution finale
    print(f"\n📊 Distribution finale par classe :")
    for event_name, event_code in event_id.items():
        n_remaining = np.sum(epochs_clean.events[:, -1] == event_code)
        print(f"   - {event_name} : {n_remaining} epochs")
    
    # Calcul des ERP
    evoked_left = epochs_clean['T1'].average()
    evoked_right = epochs_clean['T2'].average()
    
    # Visualisation
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    mne.viz.plot_compare_evokeds(
        {'Main Gauche': evoked_left, 'Main Droite': evoked_right},
        picks=['C3'], axes=axes[0], show=False, title='Canal C3'
    )
    axes[0].axvline(0, color='red', linestyle='--', alpha=0.5)
    axes[0].grid(True, alpha=0.3)
    
    mne.viz.plot_compare_evokeds(
        {'Main Gauche': evoked_left, 'Main Droite': evoked_right},
        picks=['C4'], axes=axes[1], show=False, title='Canal C4'
    )
    axes[1].axvline(0, color='red', linestyle='--', alpha=0.5)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('figures/erp_c3_c4_final.png', dpi=150)
    plt.show()
    
    print("\n✅ Figure sauvegardée : figures/erp_c3_c4_final.png")
    print("✅ epochs_clean, evoked_left, evoked_right créés")
else:
    print("\n❌ ERREUR : Aucune epoch conservée même avec seuil 1000 µV")
    print("   Vérifie la qualité de ton signal ou reprends depuis le début")

In [ ]:
print("\n" + "="*60)
print("ÉTAPE 7 : SAUVEGARDE FINALE")
print("="*60)

epochs_clean.save('reports/epochs_clean.fif', overwrite=True)
evoked_left.save('reports/evoked_left.fif', overwrite=True)
evoked_right.save('reports/evoked_right.fif', overwrite=True)

print("✅ Epochs sauvegardées : reports/epochs_clean.fif")
print("✅ Evoked sauvegardés")

# Résumé JSON
summary = {
    'n_initial': n_before,
    'n_final': n_after,
    'rejection_rate': f"{(n_before-n_after)/n_before*100:.1f}%",
    'n_left': int(np.sum(epochs_clean.events[:, -1] == 1)),
    'n_right': int(np.sum(epochs_clean.events[:, -1] == 2)),
    'sfreq': epochs_clean.info['sfreq'],
    'time_window': [tmin, tmax]
}

import json
with open('reports/preprocessing_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("✅ Résumé sauvegardé : reports/preprocessing_summary.json")

print("\n" + "="*60)
print("🎉 NOTEBOOK 02 TERMINÉ !")
print("="*60)
print(f"\n📊 RÉSULTAT FINAL :")
print(f"   - {summary['n_final']} epochs propres")
print(f"   - {summary['n_left']} main gauche")
print(f"   - {summary['n_right']} main droite")
print(f"   - Taux de rejet : {summary['rejection_rate']}")
print("\n➡️ Prochain : Notebook 03 - Visualisation avancée")

In [ ]:
# ============================================================================
# DIAGNOSTIC VISUEL COMPLET (sans sauvegarde)
# ============================================================================

print("\n" + "="*60)
print("DIAGNOSTIC VISUEL - Toutes les courbes")
print("="*60)

# 1. Afficher les drop logs
print("\n📋 Visualisation des epochs rejetées :")
epochs.plot_drop_log()
plt.show()

# 2. Vérifier l'amplitude des premières époques
print("\n📊 Visualisation des 5 premières époques :")
for i in range(min(5, len(epochs))):
    fig = epochs[i].plot(n_channels=10, scalings='auto', 
                         title=f'Epoch {i} - Avant rejet')
    plt.show()
    plt.close(fig)

# 3. Afficher l'amplitude maximale par epoch
print("\n📊 Histogramme des amplitudes maximales :")
max_amps = []
for i in range(len(epochs)):
    data = epochs.get_data()[i]
    max_amp = np.max(np.abs(data)) * 1e6
    max_amps.append(max_amp)

plt.figure(figsize=(10, 5))
plt.hist(max_amps, bins=30, color='steelblue', edgecolor='black')
plt.axvline(x=150, color='red', linestyle='--', linewidth=2, label='Seuil 150 µV')
plt.axvline(x=500, color='orange', linestyle='--', linewidth=2, label='Seuil 500 µV')
plt.axvline(x=1000, color='green', linestyle='--', linewidth=2, label='Seuil 1000 µV')
plt.xlabel('Amplitude maximale (µV)')
plt.ylabel("Nombre d'epochs")
plt.title("Distribution des amplitudes maximales par epoch")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\n   Amplitude moyenne : {np.mean(max_amps):.1f} µV")
print(f"   Amplitude médiane : {np.median(max_amps):.1f} µV")
print(f"   Amplitude max : {np.max(max_amps):.1f} µV")
print(f"   Amplitude min : {np.min(max_amps):.1f} µV")

In [ ]:
# ============================================================================
# TEST DES SEUILS - Comparaison visuelle
# ============================================================================

print("\n" + "="*60)
print("COMPARAISON DES SEUILS DE REJET")
print("="*60)

seuils = [150, 300, 500, 1000, 2000]
resultats = []

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, seuil in enumerate(seuils):
    if idx >= len(axes):
        break
    
    reject_criteria = dict(eeg=seuil * 1e-6)
    epochs_test = epochs.copy().drop_bad(reject=reject_criteria, verbose=False)
    n_after = len(epochs_test)
    taux = (len(epochs) - n_after) / len(epochs) * 100
    
    resultats.append({'seuil': seuil, 'n_epochs': n_after, 'taux_rejet': taux})
    
    # Afficher un résumé
    axes[idx].bar(['Conservées', 'Rejetées'], [n_after, len(epochs)-n_after], 
                  color=['green', 'red'])
    axes[idx].set_title(f'Seuil {seuil} µV\n{n_after}/{len(epochs)} epochs ({taux:.1f}% rejet)')
    axes[idx].set_ylabel("Nombre d'epochs")
    axes[idx].grid(True, alpha=0.3)

# Cacher le dernier subplot vide
if len(seuils) < len(axes):
    axes[-1].set_visible(False)

plt.tight_layout()
plt.show()

# Afficher le tableau des résultats
print("\n📊 Tableau récapitulatif :")
print("-" * 50)
print(f"{'Seuil (µV)':<12} {'Epochs gardées':<15} {'Taux rejet (%)':<15}")
print("-" * 50)
for r in resultats:
    print(f"{r['seuil']:<12} {r['n_epochs']:<15} {r['taux_rejet']:<15.1f}")
print("-" * 50)

# Suggestion automatique
meilleur_seuil = None
for r in resultats:
    if r['n_epochs'] > len(epochs) * 0.7:  # Garde au moins 70% des epochs
        meilleur_seuil = r['seuil']
        break

if meilleur_seuil:
    print(f"\n✅ Seuil recommandé : {meilleur_seuil} µV")
else:
    print("\n⚠️ Aucun seuil ne garde assez d'epochs, essayez 2000 µV")

In [ ]:
# ============================================================================
# SAUVEGARDE FINALE - NOTEBOOK 02
# ============================================================================

print("\n" + "="*60)
print("SAUVEGARDE FINALE")
print("="*60)

# Vérifier que les variables existent
if 'epochs_clean' not in dir():
    print("⚠️ epochs_clean non défini ! Application d'un seuil par défaut...")
    reject_criteria = dict(eeg=500e-6)
    epochs_clean = epochs.copy().drop_bad(reject=reject_criteria, verbose=False)
    print(f"✅ {len(epochs_clean)} epochs conservées avec seuil 500 µV")

if 'evoked_left' not in dir() or 'evoked_right' not in dir():
    print("⚠️ Calcul des evoked...")
    evoked_left = epochs_clean['T1'].average()
    evoked_right = epochs_clean['T2'].average()

# Sauvegarde des fichiers
epochs_clean.save('reports/epochs_clean.fif', overwrite=True)
print("✅ Epochs sauvegardées : reports/epochs_clean.fif")

evoked_left.save('reports/evoked_left.fif', overwrite=True)
evoked_right.save('reports/evoked_right.fif', overwrite=True)
print("✅ Evoked sauvegardés : reports/evoked_left.fif, reports/evoked_right.fif")

# Résumé JSON
summary = {
    'sujet': 1,
    'n_epochs_initial': len(epochs),
    'n_epochs_final': len(epochs_clean),
    'taux_conservation': f"{len(epochs_clean)/len(epochs)*100:.1f}%",
    'n_main_gauche': int(np.sum(epochs_clean.events[:, -1] == 1)),
    'n_main_droite': int(np.sum(epochs_clean.events[:, -1] == 2)),
    'frequence_echantillonnage': epochs_clean.info['sfreq'],
    'duree_epoch': 5.0,  # -1s à +4s
    'canaux_utilises': len(epochs_clean.ch_names),
    'filtres_appliques': ['passe_bande_1_40Hz_FIR', 'notch_50Hz_FIR', 'average_reference']
}

import json
with open('reports/preprocessing_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("✅ Résumé sauvegardé : reports/preprocessing_summary.json")

In [ ]:
# ============================================================================
# RAPPORT FINAL - RÉCAPITULATIF COMPLET
# ============================================================================

print("\n" + "="*60)
print("📊 RAPPORT FINAL DU PRETRAITEMENT")
print("="*60)

print("""
┌─────────────────────────────────────────────────────────────────────────┐
│                        PIPELINE DE PRETRAITEMENT                         │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  ✅ ÉTAPE 1 : Filtrage passe-bande 1-40 Hz (FIR)                        │
│     - Suppression dérive basse fréquence (<1 Hz)                         │
│     - Suppression bruit musculaire (>40 Hz)                              │
│                                                                          │
│  ✅ ÉTAPE 2 : Notch filter 50 Hz (FIR)                                   │
│     - Suppression artefact réseau électrique                             │
│                                                                          │
│  ✅ ÉTAPE 3 : Average reference                                          │
│     - Standardisation du signal                                          │
│     - Indépendant du choix de référence initial                          │
│                                                                          │
│  ✅ ÉTAPE 4 : ICA (20 composantes)                                       │
│     - Séparation des sources (cerveau/yeux/muscles)                      │
│                                                                          │
│  ✅ ÉTAPE 5 : Epoching [-1s, +4s]                                        │
│     - Découpage par essai                                                │
│     - Baseline sur [-1s, 0]                                             │
│                                                                          │
│  ✅ ÉTAPE 6 : Rejet d'artefacts                                          │
│     - Seuil d'amplitude personnalisé                                     │
│                                                                          │
└─────────────────────────────────────────────────────────────────────────┘
""")

print(f"\n📊 STATISTIQUES FINALES :")
print(f"   ├─ Sujet traité : 1")
print(f"   ├─ Epochs initiales : {summary['n_epochs_initial']}")
print(f"   ├─ Epochs finales : {summary['n_epochs_final']}")
print(f"   ├─ Taux conservation : {summary['taux_conservation']}")
print(f"   ├─ Main gauche : {summary['n_main_gauche']} epochs")
print(f"   ├─ Main droite : {summary['n_main_droite']} epochs")
print(f"   ├─ Fréquence : {summary['frequence_echantillonnage']} Hz")
print(f"   └─ Canaux EEG : {summary['canaux_utilises']}")

print("\n📁 FICHIERS GÉNÉRÉS :")
print("   ├─ reports/epochs_clean.fif (données prêtes pour classification)")
print("   ├─ reports/evoked_left.fif (moyenne main gauche)")
print("   ├─ reports/evoked_right.fif (moyenne main droite)")
print("   ├─ reports/preprocessing_summary.json (métadonnées)")
print("   └─ figures/*.png (13 figures de visualisation)")